# SCR-Screen walkthrough

This notebook shows what SCR-Screen does, step by step:

1. SCR for a single plant
2. Why nearby plants must be screened together (WSCR)
3. **Direct mode:** screen plants from a CSV file and create a report
4. **Network mode:** calculate short-circuit MVA from a network model (classical and IEC 60909)
5. Scan every bus of the IEEE 39-bus test system
6. The same results from the command line

> **Screening only.** SCR-Screen gives screening-level indicators of system strength. It does not replace interconnection studies, detailed positive-sequence or EMT analysis, or the transmission provider's requirements. All data in this notebook is illustrative.

In [1]:
from pathlib import Path

# Works whether the notebook is run from the project folder or from examples/
EXAMPLES = Path("examples") if Path("examples/plants_template.csv").exists() else Path(".")
OUTPUT = Path("notebook_output")
OUTPUT.mkdir(exist_ok=True)
print("Examples folder:", EXAMPLES.resolve())

Examples folder: /home/claude/scr-screen/examples


## 1. SCR for a single plant

SCR = short-circuit MVA at the point of interconnection (without the plant's own contribution) ÷ plant rating.

A 100 MW plant at a bus with 500 MVA of short-circuit strength:

In [2]:
from scr_screen.metrics import scr, wscr, classify

value = scr(500, 100)
print(f"SCR = {value:.3f}  ->  flag: {classify(value).value}")

SCR = 5.000  ->  flag: none


## 2. Why nearby plants must be screened together (WSCR)

Two plants that each look strong can be weak together, because they share the same grid strength. WSCR (weighted SCR, from ERCOT and NERC) accounts for this:

WSCR = Σ(SCMVAᵢ × Pᵢ) ÷ (ΣPᵢ)²

In [3]:
individual = [scr(1000, 200), scr(600, 100)]
together = wscr([1000, 600], [200, 100])

print(f"Plant 1 alone: SCR {individual[0]:.2f} ({classify(individual[0]).value})")
print(f"Plant 2 alone: SCR {individual[1]:.2f} ({classify(individual[1]).value})")
print(f"Together:      WSCR {together:.3f} ({classify(together).value})")

Plant 1 alone: SCR 5.00 (none)
Plant 2 alone: SCR 6.00 (none)
Together:      WSCR 2.889 (weak)


## 3. Direct mode: plants from a CSV file

Fill in a CSV with one row per plant (create a blank one with `scr-screen template plants.csv`). Required columns: `plant_id`, `scmva`, `rating_mw`. Optional: `rating_mva`, `group`, `poi_name`. Plants that share a `group` are also screened together with WSCR.

In [4]:
from scr_screen.io import read_plants_csv, write_results_csv
from scr_screen.screening import screen
from scr_screen.report import write_html_report

plants, poi_names = read_plants_csv(EXAMPLES / "plants_template.csv")
result = screen(plants, poi_names=poi_names)

for p in result.plants:
    print(f"{p.plant_id:<8} SCMVA {p.scmva:>8.1f}  rating {p.rating:>6.1f} MW  SCR {p.scr:.3f}  {p.flag}")
for g in result.groups:
    print(f"Group {g.group}: WSCR {g.wscr:.3f} ({g.flag}) for {', '.join(g.plant_ids)}")

PV-A     SCMVA   1000.0  rating  200.0 MW  SCR 5.000  none
BESS-B   SCMVA    600.0  rating  100.0 MW  SCR 6.000  none
Wind-C   SCMVA    485.9  rating  100.0 MW  SCR 4.859  none
Group G1: WSCR 2.889 (weak) for PV-A, BESS-B


Save the results and a self-contained HTML report (open it in any browser; print to PDF if needed):

In [5]:
report = write_html_report(result, OUTPUT / "direct_mode_report.html", title="Direct-mode example")
table = write_results_csv(result, OUTPUT / "direct_mode_results.csv")
print("Report:", report)
print("Results:", table)

Report: notebook_output/direct_mode_report.html
Results: notebook_output/direct_mode_results.csv


## 4. Network mode: short-circuit MVA from a network model

SCR-Screen can calculate the short-circuit MVA itself from a [pandapower](https://www.pandapower.org/) network, using either method:

| Method | Pre-fault voltage | Typical use |
|---|---|---|
| **Classical flat-start** (default) | 1.0 pu, no correction factors | Common U.S. planning practice |
| **IEC 60909** (via pandapower) | c = 1.1 (max case) plus impedance correction factors | IEC practice |

A small test network: a 138 kV source (1000 MVA, R/X 0.1) feeding a point of interconnection through a 50 km line. A 100 MW solar plant is connected at the POI.

In [6]:
import pandapower as pp

net = pp.create_empty_network()
src = pp.create_bus(net, vn_kv=138, name="Source")
poi = pp.create_bus(net, vn_kv=138, name="POI")
pp.create_ext_grid(net, src, s_sc_max_mva=1000, rx_max=0.1, s_sc_min_mva=800, rx_min=0.1)
pp.create_line_from_parameters(net, src, poi, length_km=50, r_ohm_per_km=0.05,
                               x_ohm_per_km=0.4, c_nf_per_km=0, max_i_ka=1)
pp.create_sgen(net, poi, p_mw=100, sn_mva=110, k=1.2)  # the solar plant (an IBR)
print(net.bus[["name", "vn_kv"]])

     name  vn_kv
0  Source  138.0
1     POI  138.0


In [7]:
import warnings
from scr_screen.network import scmva

warnings.simplefilter("ignore", FutureWarning)  # harmless notice from inside pandapower

classical = scmva(net, buses=poi, method="classical").scmva[poi]
iec = scmva(net, buses=poi, method="iec60909").scmva[poi]
print(f"Classical flat-start: {classical:.2f} MVA  ->  SCR {classical / 100:.3f}")
print(f"IEC 60909 (max):      {iec:.2f} MVA  ->  SCR {iec / 100:.3f}")
print(f"IEC is {100 * (iec / classical - 1):.1f}% higher on this network")

Classical flat-start: 485.86 MVA  ->  SCR 4.859
IEC 60909 (max):      509.68 MVA  ->  SCR 5.097
IEC is 4.9% higher on this network


**Note:** the solar plant's own fault current is **excluded automatically**, as NERC's SCR definition requires. Including it would overstate the site's strength:

In [8]:
from scr_screen.network import scmva_iec

with_ibr = scmva_iec(net, buses=poi, exclude_ibr=False).scmva[poi]
print(f"IEC with the IBR wrongly included: {with_ibr:.2f} MVA  ->  SCR {with_ibr / 100:.3f}  (too optimistic)")

IEC with the IBR wrongly included: 641.68 MVA  ->  SCR 6.417  (too optimistic)


## 5. Scan every bus of the IEEE 39-bus test system

The public IEEE 39-bus case has no short-circuit data, so `ieee39_assumed.json` adds **assumed typical values** (generator X''d = 0.20 pu, R/X = 0.05, rating = max MW ÷ 0.85). Results demonstrate the tool, not a real system.

Which buses could host a 1,500 MW plant?

In [9]:
from scr_screen.network import scan_buses

ieee39 = pp.from_json(str(EXAMPLES / "ieee39_assumed.json"))
flat = scan_buses(ieee39, plant_mw=1500, method="classical")
iec_rows = {r.bus: r for r in scan_buses(ieee39, plant_mw=1500, method="iec60909")}

print("Weakest five buses (IEEE bus number = pandapower bus + 1):")
print(f"{'IEEE bus':>8} {'SCR classical':>14} {'flag':>10} {'SCR IEC':>9} {'flag':>10}")
for r in flat[-5:]:
    i = iec_rows[r.bus]
    print(f"{r.bus + 1:>8} {r.scr:>14.2f} {r.flag:>10} {i.scr:>9.2f} {i.flag:>10}")

Weakest five buses (IEEE bus number = pandapower bus + 1):
IEEE bus  SCR classical       flag   SCR IEC       flag
      29           3.04       none      3.41       none
       1           2.94       weak      3.24       none
       9           2.86       weak      3.16       none
      28           2.31       weak      2.57       weak
      12           1.89  very weak      2.19       weak


On this system the two methods disagree on the flag at several buses. That is why SCR-Screen always states which method produced each result.

## 6. The same from the command line

Everything above is available without Python code. In a terminal (with the project environment active):

```
scr-screen template plants.csv
scr-screen run plants.csv --report report.html --out results.csv
scr-screen scan examples/ieee39_assumed.json --plant-mw 1500 --report scan.html
scr-screen scan examples/ieee39_assumed.json --plant-mw 1500 --method iec60909
```

Run `scr-screen --help` for all options (rating basis, thresholds, report title).

In [10]:
from scr_screen.cli import main

exit_code = main(["scan", str(EXAMPLES / "ieee39_assumed.json"), "--plant-mw", "1500",
      "--report", str(OUTPUT / "ieee39_scan.html")])
print("Exit code:", exit_code, "(0 means success)")

SCR-Screen v0.1.0.dev0 | basis MW | source: network model (ieee39_assumed.json), classical flat-start (1.0 pu)
Plant                     SCMVA    Rating      SCR  Flag
Bus 29                   8971.5    1500.0    5.981  none
Bus 15                   8865.0    1500.0    5.910  none
Bus 38                   8809.0    1500.0    5.873  none
Bus 1                    8804.7    1500.0    5.870  none
Bus 16                   7427.7    1500.0    4.952  none
Bus 5                    7358.9    1500.0    4.906  none
Bus 2                    7296.9    1500.0    4.865  none
Bus 24                   7090.5    1500.0    4.727  none
Bus 4                    7064.5    1500.0    4.710  none
Bus 9                    6969.4    1500.0    4.646  none
Bus 18                   6893.8    1500.0    4.596  none
Bus 3                    6841.8    1500.0    4.561  none
Bus 21                   6785.0    1500.0    4.523  none
Bus 10                   6712.9    1500.0    4.475  none
Bus 31                   6675.1   

## Limitations

- SCR, WSCR and the flag bands are **screening indicators**. There is no universal weak-grid threshold; the default bands (below 2 very weak, 2 to 3 weak) come from HVDC planning practice and are user-configurable.
- SCR-based metrics assume **grid-following** inverters; they do not replace the inverter manufacturer's minimum SCR.
- The classical method in v0.1 ignores loads, shunts and line charging, uses nominal transformer taps, and does not yet model three-winding transformers.
- WSCR assumes the plants in a group are fully interacting.

**References:** NERC, *Integrating Inverter-Based Resources into Low Short Circuit Strength Systems* (2017); NERC, *Short-Circuit Modeling and System Strength* (2018).